In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import linemerge

from geofeatureviz.data import anki_connector, loader, path_settings, river_preprocessor

# Rivers data creation
This notebook explains how the data collection for the German rivers that I want to learn in my Anki deck worked, since it was less straight-forward than expected.

At first, I get all the River names from my Anki deck for which I need the geographical information. The geographical information will be gathered from OSM using the Overpass API; and for some rivers, the names in OSM are in a different language, so they have to be changed so that they can be found in the API-request.

## River names

In [ ]:
deck_name = "Allgemeinwissen::01 🟦 Geografie 🌍::01.04 Flüsse Deutschlands 🏞️🇩🇪"
anki_rivers_df = anki_connector.deck_to_df(deck_name)
anki_rivers_df = anki_rivers_df[anki_rivers_df["NoteType"] == "Adrian::Flüsse"]

# Some rivers have different names at OSM (mainly language reasons)
anki_rivers_df["osm_name"] = anki_rivers_df["Flussname"].replace(
    {
        "Donau": "Danube",
        "Elde": "Elde-Müritz-Wasserstraße",
        "Eger": "Ohře",
        "Mosel": "La Moselle",
        "Oder": "Odra",
        "Saar": "La Sarre",
    }
)
anki_rivers_df = anki_rivers_df.sort_values(by="osm_name")
anki_rivers_df = anki_rivers_df.reset_index(drop=True)

regex = [f"^{f}$" for f in anki_rivers_df["osm_name"]]
regex = "|".join(regex)

overpass_query_names = f"""[out:json][timeout:300];
relation["waterway"="river"]["name"~"{regex}"];
out body;"""
print(overpass_query_names)

I will use this query to gather river information from OSM using the Overpass API. Since the query is rather long, it might fail; but after some retries, it always works for me. I saved the obtained JSON-file (which is not a GeoJSON!) to avoid requesting the API.

Using the names to get the rivers can be ambiguous (e.g., different rivers having the same name). OSM handles this by giving unique IDs to all objects. With this query, our goal is to find out the IDs of all rivers that we want in our dataset. Therefore, the requested data does not contain the geometries yet (I use `out body;` instead of `out geom;`).


**Side note:** the online tool [Overpass Turbo](https://overpass-turbo.eu/) allows easy access to the Overpass API. You can paste the query there and then obtain a GeoJSON-file via *Export -> GeoJSON -> Download*.

In [ ]:
api_handler_river_name = loader.OverpassAPIHandler(
    query=overpass_query_names,
    file_path=path_settings.data_raw_dir
    / "osm_rivers"
    / "german_rivers_body_name.json",
)
river_name_json = api_handler_river_name.get()

river_name_df = api_handler_river_name.parse_json(river_name_json).sort_values(
    by="name"
)

river_name_df

## River IDs

First, we take a look at the river names that appear more than once in our API-response. From a manual inspection of them on openstreetmap.org, I identified the ones that I don't want. I remove them from the list and check that all remaining rivers are the ones we also have in the Anki-Deck.

In [ ]:
# show duplicates
river_name_df[river_name_df.duplicated(subset="name", keep=False)]

In [ ]:
unwanted_ids = [15074211, 15347024, 5213209, 6799134, 7298530]
river_name_df = river_name_df[~river_name_df["id"].isin(unwanted_ids)]
river_name_df = river_name_df.reset_index(drop=True)
assert all(river_name_df.name == anki_rivers_df.osm_name)

river_name_df

From this data frame we can get the relation IDs to make a new API-request; while this is actually not necessary (we could just remove the unwanted lines from our data frame collected from the names), I think it is the cleaner way to have a clean API-request with only the IDs.

In [ ]:
river_ids_str = ";".join([f"relation({id})" for id in river_name_df["id"]])
overpass_query_river_ids = f"""[out:json][timeout:300];
({river_ids_str};);
out body;"""
api_handler_river_id = loader.OverpassAPIHandler(
    query=overpass_query_river_ids,
    file_path=path_settings.data_raw_dir / "osm_rivers" / "german_rivers_body_id.json",
)
river_id_json = api_handler_river_id.get()
river_id_df = api_handler_river_id.parse_json(river_id_json)

river_id_df = river_id_df.set_index("id")

## River members
Still, we didn't ask for the geometries in our response yet. A *relation* like a river consists of several *members*, which together form the river. In general, river relations consist of several small parts, each having their own geometry. However, my preferred representation would be a single line geometry for each river. Even though this loses some information (e.g., a river delta consisting of several streams), but it simplifies the geometry and the further work with the river geometry.

To achieve this, we have to do a lot of preprocessing. To gain all information about each member of each relation, we need to do a request for each member (and not a single one only for the relation). With our current request, we can find out the members of a relation and fetch information about the members, where we include the geometries.

In [ ]:
overpass_query_members = """[out:json][timeout:300];
({};);
out geom;"""

member_gdfs = []
for relation in river_id_df.itertuples(name="Relation"):
    members = [m for m in relation.members if m["type"] == "way"]

    member_refs_str = ";".join([f"way({m['ref']})" for m in members])
    query = overpass_query_members.format(member_refs_str)

    # make api request / load old request
    api_handler_members = loader.OverpassAPIHandler(
        query=query,
        file_path=path_settings.data_raw_dir
        / "osm_rivers"
        / "river_members"
        / f"{relation.name}_{relation.Index}_members.json",
    )
    # parse to GeoDataFrame
    members_json = api_handler_members.get()
    member_gdf = api_handler_members.parse_json(members_json)
    member_gdf.set_crs("EPSG:4326")
    # set role of members in relation
    member_roles_dict = {m["ref"]: m["role"] for m in members if m["role"] != ""}
    member_roles = [member_roles_dict.get(ref, "") for ref in member_gdf["id"]]
    member_gdf["role"] = member_roles
    # add some info about the relation to the member_gdf for easier access later
    member_gdf["relation_id"] = relation.Index
    member_gdf["relation_name"] = relation.name
    member_gdf = member_gdf.set_index("id")

    member_gdfs.append(member_gdf)

## Preprocessing
Here began the tedious part: creating a single line string from all members of a river relation.

The most obvious thing is to do the *shapely*-operation `linemerge`. However, for a lot of rivers, this returned a MultiLineString because of different reasons:
- side streams
- branches
- unclear sources

and many more. It is possible to clean up the member data frames (e.g., remove side streams), which already helped a lot. After that, only manual inspection helped to get rid of members or nodes that I didn't want in my data set. This manual inspection consisted of creating an inspection figure, where I could find the member ID's where merging the lines failed. on openstreetmap.org, I took a look at these members. In most cases, it was sufficient to just exclude this members, sometimes it was necessary to remove single nodes from the members. I saved the results of the manual inspection in a config file.

Here, all this is already applied under the hood in *river_preprocessing.py*, so in theory, all rivers should consist of a single LineString afterward.

In [ ]:
river_id_to_geom = {}
for member_gdf in member_gdfs:
    relation_id = member_gdf["relation_id"].iloc[0]
    relation_name = member_gdf["relation_name"].iloc[0]

    member_gdf = river_preprocessor.prep_member_gdf(member_gdf)
    geom = river_preprocessor.member_gdf_to_linestring(member_gdf)

    if geom.geom_type == "MultiLineString":
        # this was used for the manual cleaning; the resulting geometries should never
        # be MultLineStrings, since it was the goal of the cleaning to prevent this
        fig, ax = river_preprocessor.inspect_river_geom(geom, member_gdf)

        fig.savefig(f"{relation_id}_{relation_name}_inspect_delete.svg")
    else:
        river_id_to_geom[relation_id] = geom

river_gdf = gpd.GeoDataFrame(
    river_id_df, geometry=list(river_id_df.index.map(river_id_to_geom)), crs="EPSG:4326"
)

# save
file_path = path_settings.data_processed_dir / "osm_10m_rivers_germany.geojson"
if not file_path.exists():
    river_gdf.to_file(file_path, driver="GeoJson")

Just to see the results of our preprocessing compared to the unprocessed data, we plot both into a single map here:

In [ ]:
river_id_to_geom_unprocessed = {
    gdf["relation_id"].iloc[0]: linemerge(gdf.geometry.to_list()) for gdf in member_gdfs
}
river_gdf_unprocessed = gpd.GeoDataFrame(
    river_id_df,
    geometry=list(river_id_df.index.map(river_id_to_geom_unprocessed)),
    crs="EPSG:4326",
)

fig, ax = plt.subplots()
river_gdf_unprocessed.plot(ax=ax, color="C1")
river_gdf.plot(ax=ax, color="C0")
plt.show()

We can see that not a lot is missing, mostly small side streams. This big things are:
- side stream of the Rhine
- delta of the Danube

For both, this is intended and follows my idea of having just a single line string for each river.

In addition to preprocessing, we can now conveniently simplify our river network:

In [ ]:
simplified_gdf = river_gdf.copy()
simplified_gdf.geometry = simplified_gdf.geometry.simplify(
    tolerance=0.03, preserve_topology=True
)
simplified_gdf.plot()